In [ ]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, precision_recall_curve, recall_score, f1_score, roc_curve, roc_auc_score

1 データの読み込み

In [ ]:
file_path = "../data/raw/train-00000-of-00001.parquet"
df_train_raw = pd.read_parquet(file_path)

In [ ]:
file_path = "../data/raw/test-00000-of-00001.parquet"
df_test_raw = pd.read_parquet(file_path)

In [ ]:
# 外れ値対策
# 数値特徴量のクリッピングの閾値
# 企業の顧客DBなどの分布と合わせた方がいいと思うが、今回は公開データなので、学習データを基準とする

dict_feature_clipping_thresholds = {
    "age":{"top":59, "bottom":27}
    ,"balance":{"top":5768, "bottom":-172}
    ,"duration":{"top":751, "bottom":35}
    ,"campaign":{"top":8, "bottom":1}
    ,"pdays":{"top":370, "bottom":79}
    ,"previous":{"top":3,"bottom":0}
}

2 lightgbm

2.1 前処理の定義

In [ ]:
lgb_col_names_categorical = [
    "job", "marital", "education", "contact", "month", "poutcome"
]

lgb_col_names_binary = [
    "default", "housing", "loan", "y"
]

lgb_col_names_object = [
    "poutcome"
]

In [ ]:
# 前処理関数
# 学習データとテストデータで同一にする

def preprocessing_lgb(df):
    
    # カテゴリー型に変更
    for col in lgb_col_names_categorical:
        df[col] = df[col].astype("category")

    # binary(yes or no)のカラムをboolに変換
    for col in lgb_col_names_binary:
        df[col] = df[col].str.lower().map({"yes": True, "no": False})

    # object型をラベルエンコーディング
    for col in lgb_col_names_object:
        df[col], _ =  pd.factorize(df[col])

    # pdaysをコンタクトフラグ(-1)と経過日数に分ける
    # コンタクトをしていない＝-1、なので、-1に該当するかどうか別カラムに分ける
    df["pdays_contact_flag"] = df_train_raw["pdays"].isin([-1])
    # -1を除いた最頻値に置き換える
    mode = df_train_raw["pdays"][~(df_train_raw["pdays"]==-1)].mode()[0]
    # print(mode)
    df["pdays"] = np.where(df["pdays"] == -1, mode, df["pdays"])

    # クリッピング
    for k in dict_feature_clipping_thresholds:
        # 上限
        df[k] = np.where(
            df[k] >= dict_feature_clipping_thresholds[k]["top"]
            ,dict_feature_clipping_thresholds[k]["top"]
            ,df[k]
        )

        # 下限
        df[k] = np.where(
            df[k] <= dict_feature_clipping_thresholds[k]["bottom"]
            ,dict_feature_clipping_thresholds[k]["bottom"]
            ,df[k]
        )


    return df

In [ ]:
# preprocessing_lgbの動作確認
df_tmp_lgb = df_train_raw.copy()
df_tmp_lgb = preprocessing_lgb(df=df_tmp_lgb)

In [ ]:
df_tmp_lgb

2.2 学習、評価

In [ ]:
# 不均衡データなので、f1 scoreをカスタムメトリクスとしてlightgbmに渡してみる

def lgb_f1_score(preds, data):
    
    labels = data.get_label()
    y_pred = [1 if i >= 0.5 else 0 for i in preds]

    score = f1_score(labels, y_pred)
    
    # (指標名, スコア値, 大きい方が良いか(True/False)) を返す
    return "f1", score, True
    

In [ ]:
def lgb_train_with_mlflow(df_train, df_test, run_name, memo):
    
    y_col = "y"

    X = df_train.drop(y_col, axis=1)
    x_cols = X.columns
    y = df_train[y_col]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    train_dataset = lgb.Dataset(X_train, label=y_train)
    val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)

    X_test = df_test.drop(y_col, axis=1)
    y_test = df_test[y_col]


    params = {
        "objective": "binary",      # 2値分類
        # "metric": "binary_logloss", # 評価指標
        "metric": "None", # カスタム指標でアーリーストップとかの判断する
        "random_state": 42,
        "verbose": -1, 
        "is_unbalance": True
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=True)
    ]

    mlflow.set_experiment(
        "banking-marketing-prediction"
    )

    # アクティブなRunが残っていれば強制終了する
    # エラー防止のため
    if mlflow.active_run():
        mlflow.end_run()

    # experiment tracking本体
    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("n_features", len(x_cols))

        mlflow.log_text(
            "\n".join(x_cols),
            "features.txt"
        )

        mlflow.set_tag("memo", memo)

        # モデルの学習

        print("train starts")
        model = lgb.train(
            params,
            train_dataset,
            valid_sets=[val_dataset], 
            callbacks=callbacks,
            feval=lgb_f1_score # カスタム指標でf1
        )

        # テストデータで予測
        print("test starts")
        y_pred_prob = model.predict(X_test)
        y_pred = [1 if i >= 0.5 else 0 for i in y_pred_prob]

        # 評価指標の計算
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy: {acc}")
        mlflow.log_metric("acc", acc)

        # precision = precision_score(y_test, y_pred, pos_label=True)
        # print(f"precision_score: {precision}")

        recall = recall_score(y_test, y_pred, pos_label=True)
        print(f"recall_score: {recall}")
        mlflow.log_metric("recall", recall)

        f1 = f1_score(y_test, y_pred, pos_label=True)
        print(f"f1_score: {f1}")
        mlflow.log_metric("f1", f1)

        # feature_importanceの取得
        importances = model.feature_importance(importance_type="gain")
        df_feature_importance = pd.DataFrame({"feature": x_cols, "importance": importances})
        df_feature_importance = df_feature_importance.sort_values(by="importance", ascending=False).reset_index(drop=True)

        path_feature_importance = "../data/tmp/feature_importance.csv"
        df_feature_importance.to_csv(path_feature_importance, index=False)
        mlflow.log_artifact(path_feature_importance)

        # ROC曲線も求める
        fpr, tpr, _ = roc_curve(df_test[y_col].astype(int), y_pred_prob)
        auc = roc_auc_score(df_test[y_col].astype(int), y_pred_prob)

        path_roc =  "../data/tmp/roc_curve.png"
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
        plt.plot([0, 1], [0, 1], "k--") # 比較用にランダム予測モデルを仮定して、対角線をひき
        plt.xlabel("False Positive Rate (1 - Specificity)")
        plt.ylabel("True Positive Rate (Sensitivity)")
        plt.title("Receiver Operating Characteristic (ROC) Curve")
        plt.legend(loc="lower right")
        plt.savefig(path_roc)
        plt.close() # ノートブック上に図の表示をしない
        mlflow.log_artifact(path_roc)

        # PR曲線も求める
        precision, recall, _ = precision_recall_curve(df_test[y_col].astype(int), y_pred_prob)
        ap_score = average_precision_score(df_test[y_col].astype(int), y_pred_prob)

        path_pr =  "../data/tmp/pr_curve.png"
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f"PR curve (ap_score = {ap_score:.3f})")
        plt.plot([0, 1], [0, 1], "k--") # 比較用にランダム予測モデルを仮定して、対角線をひき
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision_Recall_curve")
        plt.legend(loc="lower right")
        plt.savefig(path_pr)
        plt.close() # ノートブック上に図の表示をしない
        mlflow.log_artifact(path_pr)
        
        


In [ ]:
# 前処理
df_train_lgb = df_train_raw.copy()
df_test_lgb = df_test_raw.copy()

df_train_lgb = preprocessing_lgb(df=df_train_lgb)
df_test_lgb = preprocessing_lgb(df=df_test_lgb)


In [ ]:
run_name = "lgb100: train_test_split(stratify=y)"
memo = "lgb：不均衡データ用のテストスプリット（trainとvalでTrue Falseの割合を合わせる）"

lgb_train_with_mlflow(
                    df_train=df_train_lgb, 
                    df_test=df_test_lgb, 
                    run_name=run_name, 
                    memo=memo
                    )

In [ ]:
# 学習履歴の確認
# CLI(/notebooksの階層で)→ mlflow ui

3 ロジスティック回帰+SVM

3.1 前処理の定義

In [ ]:
logistic_svm_col_names_one_hot_encoding = [
    "job", "marital", "education", "contact", "month", "poutcome"
]

logistic_svm_col_names_binary = [
    "default", "housing", "loan", "y"
]

In [ ]:
def preprocessing_logistic_svm(df):
    
    # onehotencoding
    # カテゴリー型に変更→onehotencdingの順番で実施

    for col in logistic_svm_col_names_one_hot_encoding:
        df[col] = df[col].astype("category")

    df = pd.get_dummies(df, columns=logistic_svm_col_names_one_hot_encoding)

    # binary(yes or no)のカラムを0,1フラグに変換
    for col in logistic_svm_col_names_binary:
        df[col] = df[col].str.lower().map({"yes": 1, "no": 0})
    
    # print(df.head(5))

    # pdaysをコンタクトフラグ(-1)と経過日数に分ける
    # コンタクトをしていない＝-1、なので、-1に該当するかどうか別カラムに分ける
    df["pdays_contact_flag"] = df_train_raw["pdays"].isin([-1])
    # -1を除いた最頻値に置き換える
    mode = df_train_raw["pdays"][~(df_train_raw["pdays"]==-1)].mode()[0]
    # print(mode)
    df["pdays"] = np.where(df["pdays"] == -1, mode, df["pdays"])

    # クリッピング,　正規化、標準化
    standard_scaler = StandardScaler()
    minmax_scaler = MinMaxScaler(feature_range=(0, 1))

    for k in dict_feature_clipping_thresholds:
        # 上限
        df[k] = np.where(
            df[k] >= dict_feature_clipping_thresholds[k]["top"]
            ,dict_feature_clipping_thresholds[k]["top"]
            ,df[k]
        )

        # 下限
        df[k] = np.where(
            df[k] <= dict_feature_clipping_thresholds[k]["bottom"]
            ,dict_feature_clipping_thresholds[k]["bottom"]
            ,df[k]
        )
    
        df[k] = minmax_scaler.fit_transform(df[[k]])
        df[k] = standard_scaler.fit_transform(df[[k]])


    return df

In [ ]:
# preprocessing_logistic_svmの動作確認
df_tmp_logsvm = df_train_raw.copy()
df_tmp_logsvm = preprocessing_logistic_svm(df=df_tmp_logsvm)

3.2 学習、評価

In [ ]:
def logistic_svm_train_with_mlflow(model_name, df_train, df_test, run_name, memo):
    
    y_col = "y"

    X_train = df_train.drop(y_col, axis=1)
    x_cols = X_train.columns
    y_train = df_train[y_col]

    X_test = df_test.drop(y_col, axis=1)
    y_test = df_test[y_col]

    
    if model_name == "svm":
        model = SVC(kernel="rbf", class_weight="balanced", random_state=42) 
        # model = SVC(kernel="rbf", probability=True, class_weight="balanced")  #計算時間がかかるので、通常時は確率を求めない。
    elif model_name == "logistic":
        model = LogisticRegression(class_weight="balanced",random_state=42)
    else:
        raise ValueError("無効なモデル名が入力されました")

    mlflow.set_experiment(
        "banking-marketing-prediction"
    )

    # アクティブなRunが残っていれば強制終了する
    # エラー防止のため
    if mlflow.active_run():
        mlflow.end_run()

    # experiment tracking本体
    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("n_features", len(x_cols))

        mlflow.log_text(
            "\n".join(x_cols),
            "features.txt"
        )

        mlflow.set_tag("memo", memo)

        # モデルの学習

        print("train starts")
        model.fit(X_train, y_train)


        # テストデータで予測
        print("test starts")
        y_pred_prob = model.predict(X_test)
        y_pred = [1 if i >= 0.5 else 0 for i in y_pred_prob]

        # y_pred_prob = model.predict_proba(X_test) #通常時は確率求めない。アンサンブル用だけ。
        # y_pred = [1 if row[1] >= 0.5 else 0 for row in y_pred_prob] # 確率の時よう

        # 評価指標の計算
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy: {acc}")
        mlflow.log_metric("acc", acc)

        # precision = precision_score(y_test, y_pred, pos_label=True)
        # print(f"precision_score: {precision}")

        recall = recall_score(y_test, y_pred, pos_label=True)
        print(f"recall_score: {recall}")
        mlflow.log_metric("recall", recall)

        f1 = f1_score(y_test, y_pred, pos_label=True)
        print(f"f1_score: {f1}")
        mlflow.log_metric("f1", f1)

        # ROC曲線も求める
        fpr, tpr, _ = roc_curve(df_test[y_col].astype(int), y_pred_prob)
        auc = roc_auc_score(df_test[y_col].astype(int), y_pred_prob)

        path_roc =  "../data/tmp/roc_curve.png"
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
        plt.plot([0, 1], [0, 1], "k--") # 比較用にランダム予測モデルを仮定して、対角線をひき
        plt.xlabel("False Positive Rate (1 - Specificity)")
        plt.ylabel("True Positive Rate (Sensitivity)")
        plt.title("Receiver Operating Characteristic (ROC) Curve")
        plt.legend(loc="lower right")
        plt.savefig(path_roc)
        plt.close() # ノートブック上に図の表示をしない
        mlflow.log_artifact(path_roc)

        # PR曲線も求める
        precision, recall, _ = precision_recall_curve(df_test[y_col].astype(int), y_pred_prob)
        ap_score = average_precision_score(df_test[y_col].astype(int), y_pred_prob)

        path_pr =  "../data/tmp/pr_curve.png"
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f"PR curve (ap_score = {ap_score:.3f})")
        plt.plot([0, 1], [0, 1], "k--") # 比較用にランダム予測モデルを仮定して、対角線をひき
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision_Recall_curve")
        plt.legend(loc="lower right")
        plt.savefig(path_pr)
        plt.close() # ノートブック上に図の表示をしない
        mlflow.log_artifact(path_pr)

In [ ]:
# 前処理
df_train_logistic_svm = df_train_raw.copy()
df_test_logistic_svm = df_test_raw.copy()

df_train_logistic_svm = preprocessing_logistic_svm(df=df_train_logistic_svm)
df_test_logistic_svm = preprocessing_logistic_svm(df=df_test_logistic_svm)

In [ ]:
# model_name = "logistic"
model_name = "svm"

run_name = f"{model_name}100:Add train_test_split(stratify=y)"
memo = f"{model_name}:不均衡データ用のテストスプリット（trainとvalでTrue Falseの割合を合わせる）"

logistic_svm_train_with_mlflow(
                    model_name = model_name,
                    df_train=df_train_logistic_svm, 
                    df_test=df_test_logistic_svm, 
                    run_name=run_name, 
                    memo=memo
                    )

In [ ]:
# 学習履歴の確認
# CLI(/notebooksの階層で)→ mlflow ui

4.アンサンブル学習用(スタッキング)

4.1 lightgbm

In [ ]:
df_train_lgb = df_train_raw.copy()
df_test_lgb = df_test_raw.copy()

df_train_lgb = preprocessing_lgb(df=df_train_lgb)
df_test_lgb = preprocessing_lgb(df=df_test_lgb)

# アンサンブル学習のlgbの予測確率出すように、前処理用のテストデータを保存しておく。
df_test_lgb.to_csv("../data/processed/test_lgb.csv", index=False, encoding="utf-8")


In [ ]:
# df_train_lgb.columns == df_test_lgb.columns

In [ ]:
 
y_col = "y"

X = df_train_lgb.drop(y_col, axis=1)
x_cols = X.columns
y = df_train_lgb[y_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_dataset = lgb.Dataset(X_train, label=y_train)
val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)

X_test = df_test_lgb.drop(y_col, axis=1)
y_test = df_test_lgb[y_col]



params = {
    "objective": "binary",      # 2値分類
    # "metric": "binary_logloss", # 評価指標
    "metric": "None", # カスタム指標でアーリーストップとかの判断する
    "random_state": 42,
    "verbose": -1, 
    "is_unbalance": True
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True)
]

model_lgb = lgb.train(
        params,
        train_dataset,
        valid_sets=[val_dataset], 
        callbacks=callbacks,
        feval=lgb_f1_score # カスタム指標でf1
    )


In [ ]:
# モデルの保存
lgb_save_path = "../models/model_lgb.txt"
model_lgb.save_model(lgb_save_path)

In [ ]:
lgb_prob = model_lgb.predict(df_test_lgb.drop("y",axis=1))

4.2 ロジスティック回帰、SVM

In [ ]:
df_train_logistic_svm = df_train_raw.copy()
df_test_logistic_svm = df_test_raw.copy()

df_train_logistic_svm = preprocessing_logistic_svm(df=df_train_logistic_svm)
df_test_logistic_svm = preprocessing_logistic_svm(df=df_test_logistic_svm)

# アンサンブル学習のlogregとsvm予測確率出すように、前処理済みのテストデータを保存しておく。
df_test_logistic_svm.to_csv("../data/processed/test_logreg_svm.csv", index=False, encoding="utf-8")

In [ ]:
y_col = "y"

X_train = df_train_logistic_svm.drop(y_col, axis=1)
x_cols = X_train.columns
y_train = df_train_logistic_svm[y_col]

X_test = df_test_logistic_svm.drop(y_col, axis=1)
y_test = df_test_logistic_svm[y_col]

model_svm = SVC(kernel="rbf", probability=True, class_weight="balanced", tol=1e-2, cache_size=1000) 
model_logreg = LogisticRegression(class_weight="balanced")



In [ ]:
print("train starts")
model_svm.fit(X_train, y_train)
model_logreg.fit(X_train, y_train)

In [ ]:
# 保存

svm_save_path = "../models/model_svm.joblib"
joblib.dump(model_svm, svm_save_path)

logreg_save_path = "../models/model_logreg.joblib"
joblib.dump(model_logreg, logreg_save_path)



In [ ]:
svm_prob = model_svm.predict_proba(X_test)
logreg_prob =  model_logreg.predict_proba(X_test)


In [ ]:
df_corr = pd.DataFrame()
df_corr["lgb"] = lgb_prob
df_corr["logreg"] = [row[1] for row in logreg_prob]
df_corr["svm"] = [row[1] for row in svm_prob]



In [ ]:
df_corr.corr()